In [ ]:
"""
Notebook driver for PASCHEN-1D postprocessing from saved run files.

Requirements:
- A folder named by run_name must exist in the project directory.
- That folder must contain saved sampled arrays and run_metadata.json
  (written automatically by run_simulation).

Use this notebook to regenerate and restyle diagnostics without rerunning
simulation.
"""

from pathlib import Path

from postprocess import (
    replot_from_saved,
    TemporalReplotStyle,
    SpatialReplotStyle,
)

# --------------------------------------------------------------------
# 1) Discover available runs (folders containing run_metadata.json)
# --------------------------------------------------------------------
project_dir = Path('.')
available_runs = sorted(
    p.name for p in project_dir.iterdir()
    if p.is_dir() and (p / 'run_metadata.json').exists()
)

print('Available runs with metadata:')
for r in available_runs:
    print(f'  - {r}')

# Set this to any previous run folder name from the list above.
run_name = 'argon_rf_L6p5cm_13p56MHz_test2'
print(f'\nSelected run_name: {run_name}')


In [ ]:
# --------------------------------------------------------------------
# 2) Postprocessing style controls (edit freely)
# --------------------------------------------------------------------
import matplotlib.pyplot as plt

# Publication-oriented save defaults
plt.rcParams.update({
    'savefig.dpi': 600,
    'figure.dpi': 150,
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'legend.fontsize': 10,
})

temporal_style = TemporalReplotStyle(
    t_unit='us',         # 's', 'ms', 'us', 'ns'
    xscale='linear',     # 'linear' or 'log'
    yscale='linear',     # 'linear' or 'log'
)

spatial_style = SpatialReplotStyle(
    x_unit='cm',         # 'm', 'cm', 'mm'
    xscale='linear',     # 'linear' or 'log'
    yscale='linear',     # 'linear' or 'log'
)

# --------------------------------------------------------------------
# 3) Choose groups/times (optional)
# --------------------------------------------------------------------

# Available Temporal Diagnostics = Literal[
#     "V_app",
#     "V_gap",
#     "I_discharge",
#     "cfl",
#     "particle_inventory",
# ]

# Available Spatial Diagnostics = Literal[
#     "ne",
#     "ni",
#     "phi",
#     "E",
#     "Gamma_i",
#     "Gamma_e",
#     "townsend_alpha",
#     "nu_i",
#     "S",
# ]

temporal_groups = (
    ('V_app', 'V_gap'),
    ('I_discharge',),
#     ('cfl',),
#     ('particle_inventory',),
)

spatial_groups = (
#     ('ne', 'ni'),
#     ('phi',),
#     ('E',),
)

averaged_spatial_groups = (
    ('ne', 'ni'),
    ('phi',),
    ('E',),
)

# Time window for temporal plots (None -> full range)
t_start = None
t_end = None

# Spatial sample times in seconds (None -> final saved snapshot only)
t_samples = None
# Example: t_samples = (5e-6, 10e-6, 15e-6)

# Averaged-spatial controls
averaged_mode = 'last_n_cycles'  # 'time_window' or 'last_n_cycles'
N_cycle_avg = 10
t_avg_start = None
t_avg_end = None
# For time-window averaging, set for example:
# averaged_mode = 'time_window'
# t_avg_start = 4.0e-6
# t_avg_end = 5.0e-6

# --------------------------------------------------------------------
# 3b) Optional per-figure text/legend overrides (edit as needed)
# --------------------------------------------------------------------
# Figure index follows creation order in this notebook run (1, 2, 3, ...).
# Leave empty {} to keep default labels/titles/legends from plotting code.
figure_overrides = {
    1: {
        'title': '',
        'xlabel': 'Time (us)',
        'ylabel': 'Voltage (kV)',
        'legend_show': True,
        'legend_loc': 'best',
        'xlim': None,
        'ylim': None,
    },
}
# figure_overrides = {}

# --------------------------------------------------------------------
# 4) Regenerate plots from saved files and auto-save publication figures
# --------------------------------------------------------------------
outdir = Path(run_name) / 'postprocess_figures'
outdir.mkdir(parents=True, exist_ok=True)

orig_show = plt.show
save_count = {'n': 0}

def _apply_overrides(fig, idx):
    cfg = figure_overrides.get(idx)
    if not cfg:
        return
    if len(fig.axes) == 0:
        return
    ax = fig.axes[0]

    if cfg.get('title') is not None:
        ax.set_title(cfg['title'])
    if cfg.get('xlabel') is not None:
        ax.set_xlabel(cfg['xlabel'])
    if cfg.get('ylabel') is not None:
        ax.set_ylabel(cfg['ylabel'])

    xlim = cfg.get('xlim', None)
    if xlim is not None and len(xlim) == 2:
        ax.set_xlim(xlim[0], xlim[1])

    ylim = cfg.get('ylim', None)
    if ylim is not None and len(ylim) == 2:
        ax.set_ylim(ylim[0], ylim[1])

    legend_show = cfg.get('legend_show', None)
    if legend_show is False:
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()
    elif legend_show is True:
        handles, labels = ax.get_legend_handles_labels()
        labels_override = cfg.get('legend_labels', None)
        loc = cfg.get('legend_loc', 'best')
        if labels_override is not None and len(labels_override) == len(handles):
            ax.legend(handles, labels_override, frameon=False, loc=loc)
        else:
            ax.legend(frameon=False, loc=loc)


def save_and_show(*args, **kwargs):
    fig = plt.gcf()
    if fig is not None and len(fig.axes) > 0:
        save_count['n'] += 1
        _apply_overrides(fig, save_count['n'])
        base = outdir / f'postprocess_fig_{save_count["n"]:02d}'
        fig.savefig(base.with_suffix('.pdf'), bbox_inches='tight')
        fig.savefig(base.with_suffix('.png'), dpi=600, bbox_inches='tight')
    return orig_show(*args, **kwargs)

plt.show = save_and_show
try:
    replot_from_saved(
        run_name=run_name,
        temporal_groups=temporal_groups,
        spatial_groups=spatial_groups,
        averaged_spatial_groups=averaged_spatial_groups,
        t_start=t_start,
        t_end=t_end,
        t_samples=t_samples,
        averaged_mode=averaged_mode,
        t_avg_start=t_avg_start,
        t_avg_end=t_avg_end,
        N_cycle_avg=N_cycle_avg,
        temporal_style=temporal_style,
        spatial_style=spatial_style,
    )
finally:
    plt.show = orig_show

print(f"Saved {save_count['n']} figure(s) to: {outdir}")
